<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/assignments/assignment_yourname_t81_558_class8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# T81-558: Anwendungen tiefer neuronaler Netzwerke
* Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
* Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

**Modul 8, Aufgabe: Feature Engineering**

**Name des Studenten: Ihr Name**

# Aufgabenanweisungen

Diese Aufgabe ähnelt Aufgabe 5, außer dass Sie zur Lösung Feature Engineering verwenden müssen. Ich stelle Ihnen einen Datensatz zur Verfügung, der die Abmessungen und die Qualität von Artikeln mit bestimmten Formen enthält. Mit den Werten „Höhe“, „Breite“, „Tiefe“, „Form“ und „Qualität“ sollten Sie versuchen, die Kosten dieser Artikel vorherzusagen. Wenn Sie das Feature Engineering richtig durchführen, sollten Sie in der Lage sein, eine sehr genaue Übereinstimmung mit der Lösungsdatei zu finden. Um die volle Punktzahl zu erhalten, sollten Ihre durchschnittlichen Kosten nicht mehr als 50 % von der Lösung abweichen. Die Autokorrektur teilt Ihnen mit, ob Sie in diesem Bereich liegen.

Alle benötigten CSV-Dateien finden Sie hier:

* [Shapes - Training](https://data.heatonresearch.com/data/t81-558/datasets/shapes-train.csv)
* [Shapes - Submit](https://data.heatonresearch.com/data/t81-558/datasets/shapes-test.csv)

Verwenden Sie die Trainingsdatei, um Ihr neuronales Netzwerk zu trainieren und Ergebnisse für die in der Test-/Übermittlungsdatei enthaltenen Daten zu übermitteln.

In [ ]:
try:
    from google.colab import drive, userdata

    drive.mount("/content/drive", force_remount=True)
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Schlüssel zur Aufgabenabgabe – wurde Ihnen in der ersten Unterrichtswoche zugesandt.
# Wenn Sie in beiden Klassen sind, ist dies derselbe Schlüssel.
if COLAB:
    # Für Colab fügen Sie zu Ihren „Geheimnissen“ hinzu (Schlüsselsymbol links)
    key = userdata.get("T81_558_KEY")
else:
    # Wenn es sich nicht um Colab handelt, geben Sie hier Ihren Schlüssel ein oder verwenden Sie eine Umgebungsvariable.
    # (das ist nur ein Beispielschlüssel, verwenden Sie Ihren)
    key = "Gx5en9cEVvaZnjhdaushddhuhhO4PsI32sgldAXj"

# Funktion zum Übermitteln von Aufgaben

Die 10 Programmieraufgaben reichen Sie elektronisch ein. Hierzu steht Ihnen die folgende Übermittlungsfunktion zur Verfügung. Mein Server führt bei jeder Aufgabe eine Grundprüfung durch und meldet sich bei Ihnen, wenn er grundlegende Probleme feststellt.

**Es ist unwahrscheinlich, dass diese Funktion geändert werden muss.**

In [ ]:
import base64
import os
import numpy as np
import pandas as pd
import requests
import PIL
import PIL.Image
import io
from typing import List, Union

# Mit dieser Funktion können Sie eine Aufgabe einreichen. Sie können eine Aufgabe so oft einreichen, wie Sie möchten, nur die endgültige
# Die Anzahl der Einreichungen. Die Parameter sind wie folgt:
# Daten – Liste von Pandas-Datenrahmen oder -Bildern.
# Schlüssel – Ihr Studentenschlüssel, der Ihnen per E-Mail zugeschickt wurde.
# Kurs – Der Kurs, an dem Sie teilnehmen, derzeit t81-558 oder t81-559.
# nein – Die Zuweisungsklassennummer sollte zwischen 1 und 10 liegen.
# Quelldatei – Der vollständige Pfad zu Ihrer Python- oder IPYNB-Datei. Der Name muss „_class1“ enthalten.
# . Die Nummer muss mit Ihrer Aufgabennummer übereinstimmen. Zum Beispiel „_class2“ für Klassenaufgabe Nr. 2.

def submit(
    data: List[Union[DataFrame, PIL.Image.Image]],
    key: str,
    course: str,
    no: int,
    source_file: str = None
) -> None:
    if source_file is None and '__file__' not in globals():
        raise Exception("Must specify a filename when in a Jupyter notebook.")
    if source_file is None:
        source_file = __file__

    suffix = f'_class{no}'
    if suffix not in source_file:
        raise Exception(f"{suffix} must be part of the filename.")

    ext = os.path.splitext(source_file)[-1].lower()
    if ext not in ['.ipynb', '.py']:
        raise Exception(f"Source file is {ext}; must be .py or .ipynb")

    with open(source_file, "rb") as file:
        encoded_python = base64.b64encode(file.read()).decode('ascii')

    payload = []
    for item in data:
        if isinstance(item, PIL.Image.Image):
            buffered = io.BytesIO()
            item.save(buffered, format="PNG")
            payload.append({'PNG': base64.b64encode(buffered.getvalue()).decode('ascii')})
        elif isinstance(item, DataFrame):
            payload.append({'CSV': base64.b64encode(item.to_csv(index=False).encode('ascii')).decode("ascii")})
        else:
            raise ValueError(f"Unsupported data type: {type(item)}")

    response = requests.post(
        "https://api.heatonresearch.com/wu/submit",
        headers={'x-api-key': key},
        json={
            'payload': payload,
            'assignment': no,
            'course': course,
            'ext': ext,
            'py': encoded_python
        }
    )

    if response.status_code == 200:
print(f"Erfolg: {response.text}")
    else:
print(f"Fehler: {response.text}")

# Aufgabe Nr. 8 Beispielcode

Der folgende Code bietet einen Ausgangspunkt für diese Zuweisung.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim

# Sie müssen Ihre Quelldatei identifizieren. (für Ihr lokales Setup ändern)
file = "/content/drive/My Drive/Colab Notebooks/assignment_yourname_t81_558_class8.ipynb"  # Google CoLab
# Datei = 'C:\\Benutzer\\jeffh\\Projekte\\t81_558_deep_learning\\Aufgaben\\Aufgabe_IhrName_Klasse8.ipynb' # Windows
# Datei = "/Benutzer/jheaton/Projekte/t81_558_deep_learning/assignments/assignment_IhrName_Klasse8.ipynb" # Mac/Linux


# Daten lesen
df_train = read_csv(
    "https://data.heatonresearch.com/data/t81-558/datasets/shapes-train.csv"
)
df_submit = read_csv(
    "https://data.heatonresearch.com/data/t81-558/datasets/shapes-test.csv"
)

# # ... setzen Sie Ihren Code fort...

# # Aufgabe einreichen

submit(source_file=file, data=[submit_df], key=key, no=8, course="t81-558")